# Query Your Baseball Data

Learn how to ask questions of your data using different query methods.

In [ ]:
import pandas as pd
import duckdb

# Load the data
df = pd.read_parquet("data/statcast_2025.parquet")
print(f"Total pitches in dataset: {len(df)}")
print(f"Columns available: {len(df.columns)}")

## Method 1: Pandas Queries (Easiest)

In [ ]:
# Query 1: Show all fastballs
fastballs = df[df['pitch_type'] == 'FF']
print(f"Total fastballs: {len(fastballs)}")
print(fastballs.head())

In [ ]:
# Query 2: Fast fastballs (> 95 mph)
fast_fastballs = df[(df['pitch_type'] == 'FF') & (df['release_speed'] > 95)]
print(f"Fastballs over 95 mph: {len(fast_fastballs)}")
print(f"Average speed: {fast_fastballs['release_speed'].mean():.1f} mph")
print(fast_fastballs[['pitcher', 'pitch_type', 'release_speed']].head())

## Method 2: SQL Queries (More Powerful)

Use DuckDB to write actual SQL queries directly on your parquet files.

In [ ]:
# SQL Query 1: Average pitch velocity by pitch type
result = duckdb.query("""
    SELECT 
        pitch_type, 
        COUNT(*) as count,
        ROUND(AVG(release_speed), 2) as avg_speed,
        ROUND(MIN(release_speed), 2) as min_speed,
        ROUND(MAX(release_speed), 2) as max_speed
    FROM df
    GROUP BY pitch_type
    ORDER BY avg_speed DESC
""").to_df()

print("Pitch Type Statistics:")
print(result)

In [ ]:
# SQL Query 2: Top 10 fastest pitchers (by average fastball speed)
result = duckdb.query("""
    SELECT 
        pitcher,
        COUNT(*) as fastball_count,
        ROUND(AVG(release_speed), 1) as avg_fastball_speed
    FROM df
    WHERE pitch_type = 'FF'
    GROUP BY pitcher
    HAVING COUNT(*) >= 10
    ORDER BY avg_fastball_speed DESC
    LIMIT 10
""").to_df()

print("Top 10 Fastest Pitchers (Fastball):")
print(result)

## Method 3: Using pandas .query() method

A cleaner pandas syntax that looks more like SQL.

In [ ]:
# Using .query() - looks like SQL but it's pandas
result = df.query("pitch_type == 'FF' and release_speed > 95")
print(f"Found {len(result)} fastballs over 95 mph")
print(result[['pitcher', 'release_speed', 'balls', 'strikes']].head())

## Common Queries You Might Want to Try

In [ ]:
# Query: Pitches thrown in high-pressure situations (0-2 count or 2-0 count)
high_pressure = df.query("(balls == 0 and strikes == 2) or (balls == 2 and strikes == 0)")
print(f"High pressure pitches: {len(high_pressure)}")
print(high_pressure['pitch_type'].value_counts())

print("\n" + "="*50 + "\n")

# Query: Most common pitch type by game state
print("Most common pitch type in 3-2 count (full count):")
full_count = df.query("balls == 3 and strikes == 2")
print(full_count['pitch_type'].value_counts().head())